In [3]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # Necessário para IterativeImputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'pandas'

 ### 1. GERADOR DE DATASET FINANCEIRO SINTÉTICO (SIMULAÇÃO REALISTA)

In [4]:
def gerar_dataset_financeiro(n_samples=3000, random_state=42):
    np.random.seed(random_state)

    # Target desbalanceado (apenas ~6% de inadimplência)
    y = np.random.choice([0, 1], size=n_samples, p=[0.94, 0.06])

    # Variáveis Numéricas
    renda_mensal = np.random.lognormal(mean=8.2, sigma=0.6, size=n_samples)
    valor_emprestimo = renda_mensal * np.random.uniform(2, 12, size=n_samples)
    score_credito = np.random.normal(loc=650 - 120*y, scale=80, size=n_samples)
    score_credito = np.clip(score_credito, 300, 850)

    # Variáveis Categóricas de Alta Cardinalidade
    cidades = [f"Cidade_{i}" for i in range(1, 401)]
    ramos = [f"Ramo_Atividade_{i}" for i in range(1, 61)]

    # Probabilidades correlacionadas ao target para simular poder preditivo
    prob_cidades = np.random.dirichlet(np.ones(len(cidades)))
    prob_ramos = np.random.dirichlet(np.ones(len(ramos)))

    codigo_cidade = np.random.choice(cidades, size=n_samples, p=prob_cidades)
    ramo_atividade = np.random.choice(ramos, size=n_samples, p=prob_ramos)
    canal_entrada = np.random.choice(['Web', 'App', 'Agencia', 'Correspondente'], size=n_samples)

    df = pd.DataFrame({
        'renda_mensal': renda_mensal,
        'valor_emprestimo': valor_emprestimo,
        'score_credito': score_credito,
        'codigo_cidade': codigo_cidade,
        'ramo_atividade': ramo_atividade,
        'canal_entrada': canal_entrada,
        'inadimplente': y
    })

    # Introduzir Missing Values (MAR/MCAR)
    mask_renda = np.random.rand(n_samples) < 0.12  # 12% missing
    mask_score = np.random.rand(n_samples) < 0.15  # 15% missing

    df.loc[mask_renda, 'renda_mensal'] = np.nan
    df.loc[mask_score, 'score_credito'] = np.nan

    return df


### 2. TRANSFORMADOR CUSTOMIZADO: SmoothedTargetEncoder

In [ ]:
class SmoothedTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Transformador de Target Encoding com Suavização Bayesiana (Smoothing)
    para mitigação de Data Leakage e Overfitting em alta cardinalidade.
    Compatible com scikit-learn Pipelines.
    """
    def __init__(self, cols=None, smoothing=10.0):
        self.cols = cols
        self.smoothing = smoothing
        self.mapping_ = {}
        self.global_mean_ = 0.0

    def fit(self, X, y):
        if y is None:
            raise ValueError("O parâmetro 'y' é obrigatório para o Target Encoding.")

        X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
        y_ser = pd.Series(y) if not isinstance(y, pd.Series) else y.copy()

        # Define colunas a codificar se não especificadas
        if self.cols is None:
            self.cols = X_df.select_dtypes(include=['object', 'category']).columns.tolist()

        self.global_mean_ = y_ser.mean()
        self.mapping_ = {}

        for col in self.cols:
            # Agrupamento por categoria: contagem e média do target
            stats = y_ser.groupby(X_df[col]).agg(['count', 'mean'])
            counts = stats['count']
            means = stats['mean']

            # Fórmula da Suavização Bayesiana: (n * mean + m * global_mean) / (n + m)
            smoothed_vals = (counts * means + self.smoothing * self.global_mean_) / (counts + self.smoothing)

            self.mapping_[col] = smoothed_vals.to_dict()

        return self

    def transform(self, X):
        X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()

        for col in self.cols:
            if col in self.mapping_:
                mapper = self.mapping_[col]
                # Mapeia categorias conhecidas; novas/unseen recebem a média global
                X_df[col] = X_df[col].map(mapper).fillna(self.global_mean_).astype(float)

        return X_df

### 3. CONSTRUÇÃO E COMPARAÇÃO DOS PIPELINES

In [ ]:
def executar_pratica():
    print(">>> 1. Gerando Dataset Financeiro com Missing & Alta Cardinalidade...")
    df = gerar_dataset_financeiro(n_samples=3500)

    X = df.drop(columns=['inadimplente'])
    y = df['inadimplente']

    num_cols = ['renda_mensal', 'valor_emprestimo', 'score_credito']
    cat_cols = ['codigo_cidade', 'ramo_atividade', 'canal_entrada']

    print(f"Formato do Dataset: {X.shape}")
    print(f"Taxa de Inadimplência: {y.mean():.2%}")
    print(f"Valores Ausentes em 'renda_mensal': {X['renda_mensal'].isna().sum()}")
    print(f"Valores Ausentes em 'score_credito': {X['score_credito'].isna().sum()}\n")


### PIPELINE 1: Proposta Avançada (MICE / KNNImputer + SmoothedTargetEncoder)

In [ ]:
 num_transformer_adv = Pipeline(steps=[
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler', StandardScaler())
    ])

    cat_transformer_adv = Pipeline(steps=[
        ('target_enc', SmoothedTargetEncoder(cols=cat_cols, smoothing=15.0))
    ])

    preprocessor_adv = ColumnTransformer(transformers=[
        ('num', num_transformer_adv, num_cols),
        ('cat', cat_transformer_adv, cat_cols)
    ])

    pipeline_avancado = Pipeline(steps=[
        ('preprocessor', preprocessor_adv),
        ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
    ])

### PIPELINE 2: Baseline Simples (SimpleImputer + OrdinalEncoder)

In [ ]:
 num_transformer_base = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer_base = Pipeline(steps=[
        ('ordinal_enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    preprocessor_base = ColumnTransformer(transformers=[
        ('num', num_transformer_base, num_cols),
        ('cat', cat_transformer_base, cat_cols)
    ])

    pipeline_baseline = Pipeline(steps=[
        ('preprocessor', preprocessor_base),
        ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
    ])

### 4. VALIDAÇÃO CRUZADA E COMPARAÇÃO DE MÉTRICAS

In [ ]:
    print(">>> 2. Executando Validação Cruzada Estratificada (5-Fold)...")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring = ['roc_auc', 'average_precision']

    scores_adv = cross_validate(pipeline_avancado, X, y, cv=cv, scoring=scoring)
    scores_base = cross_validate(pipeline_baseline, X, y, cv=cv, scoring=scoring)

    print("\n" + "="*70)
    print("RESULTADOS DA VALIDAÇÃO CRUZADA (MÉDIAS)")
    print("="*70)
    print(f"PROPOSTA AVANÇADA (KNNImputer + SmoothedTargetEncoder):")
    print(f"  • ROC-AUC:            {scores_adv['test_roc_auc'].mean():.4f} (+/- {scores_adv['test_roc_auc'].std():.4f})")
    print(f"  • PR-AUC (Avg Prec):  {scores_adv['test_average_precision'].mean():.4f} (+/- {scores_adv['test_average_precision'].std():.4f})")
    print("-" * 70)
    print(f"BASELINE SIMPLES (SimpleImputer + OrdinalEncoder):")
    print(f"  • ROC-AUC:            {scores_base['test_roc_auc'].mean():.4f} (+/- {scores_base['test_roc_auc'].std():.4f})")
    print(f"  • PR-AUC (Avg Prec):  {scores_base['test_average_precision'].mean():.4f} (+/- {scores_base['test_average_precision'].std():.4f})")
    print("="*70)

In [ ]:
if __name__ == "__main__":
    executar_pratica()